**Last updated: 2026-11-24** - First version.

# Circuit model from the HFSS + EPR runs

Reads what `qubits_couplers.ipynb` saves from its HFSS eigenmode + EPR runs (`<run>_modes.csv`, `<run>_chi.csv`, `<run>_convergence.csv`) and turns each three-element run into a circuit model that can be swept without more simulation.

For each run (Q-TC-Q or TC-Q-TC) a model of three tunable transmons with charge-charge couplings is fitted to reproduce exactly what pyEPR found: the three dressed frequencies, the three anharmonicities and the three cross-Kerr shifts. That is nine numbers for nine unknowns (E_J and E_C of each element, and the coupling of each pair), so the fit is exact. It gives:

* **Couplings** g of every pair, including the **direct** qubit-qubit (or coupler-coupler) coupling, which the cross-Kerr inversion in `qubits_couplers.ipynb` cannot separate from the coupling through the middle element.
* **Coupler flux sweeps**: coupler frequency, qubit shifts, static ZZ and the effective qubit-qubit coupling g_eff against the coupler's flux, with the idle point and any zero crossings.
* **Retuning**: how much each element's C_sigma (or, for a coupler, its junction) has to change to land on its target frequency, the anharmonicity it then has, and the HFSS calibration factors that would make `qubits_couplers.ipynb` size the pads that way.
* **Convergence and T1**: each run's last adaptive steps with an estimate of how far each mode still had to move, and the control-line T1 limits if the run had lossy ports.

**Kernel:** `qiskit_metal_kernel` (the qiskit_metal_env conda environment, which has scqubits). scqubits supplies each transmon's levels and charge matrix elements; the coupled three-element Hamiltonian is built here from them, because scqubits 4.3.1's `HilbertSpace` fails with the scipy (1.17) in that environment.

Edit the *Settings* cell, then **Restart & Run All**.

## Settings

In [ ]:
%matplotlib inline
import glob
import itertools
import os
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scqubits as scq
from scipy.optimize import brentq, least_squares, minimize_scalar

scq.settings.PROGRESSBAR_DISABLED = True
warnings.filterwarnings("ignore", category=FutureWarning)

# =====================================================================
#  SETTINGS
# =====================================================================
# Folder of HFSS results written by qubits_couplers.ipynb (hfss_results_dir
# there). None: the most recently written hfss_results*/qubits_couplers.
results_dir = None

# Design values, used when the results folder has no junctions.csv (written
# by qubits_couplers.ipynb from 2026-11-24). Copied from its parameters and
# junction table; keep them in step with it.
DESIGN_COLUMNS = ["target (GHz)", "finger (um)", "C_sigma (fF)", "C_j (fF)", "E_J (GHz)"]
design_fallback = {
    # name: (target f01, finger width, C_sigma as HFSS sees it, C_j, E_J of the SQUID)
    "Q0": (5.05, 0.205, 115.66, 4.18, 20.365), "Q1": (4.80, 0.205, 128.49, 4.18, 20.365),
    "Q2": (4.55, 0.205, 143.52, 4.18, 20.365), "Q3": (4.80, 0.205, 128.49, 4.18, 20.365),
    "Q4": (5.05, 0.205, 115.66, 4.18, 20.365),
    "TC0": (6.60, 0.395, 133.52, 8.06, 39.239), "TC1": (6.35, 0.395, 144.50, 8.06, 39.239),
    "TC2": (6.35, 0.395, 144.50, 8.06, 39.239), "TC3": (6.60, 0.395, 133.52, 8.06, 39.239),
}
# HFSS calibration factors in qubits_couplers.ipynb at the time of the runs
# (qubit_csigma_hfss_factor / coupler_csigma_hfss_factor, before the
# hfss_eps_scale substrate rescaling, which is unchanged by retuning).
qubit_csigma_hfss_factor = 1.12
coupler_csigma_hfss_factor = 1.14
jj_grid_um = 0.005  # finger widths snap to this (SQP3.1)
dfm_finger_um = 0.200  # SE.DJ.W.1.DFM

# Model size. Each transmon: charge basis |n| <= ncut, kept to its lowest
# levels_kept levels in the coupled Hamiltonian (levels_kept ** 3 states).
# 5 levels resolve the kHz-scale ZZ; check by raising it to 6.
ncut = 15
levels_kept = 5

# Mode labelling: a dressed state is trusted as "bare state X" only if it
# holds at least this much of X. Sweep points where any labelled state falls
# below it (the coupler crossing a qubit) are left blank.
min_label_overlap = 0.6
# A run is skipped if any mode's largest junction participation is under this
# (the mode is shared between elements; see qubits_couplers.ipynb).
min_participation = 0.8

# Coupler flux sweep, in flux quanta (0 = maximum frequency, the idle point).
flux_points = np.round(np.linspace(0.0, 0.45, 91), 6)
geff_flux_points = np.round(np.linspace(0.0, 0.45, 19), 6)  # numerical g_eff is slower

# Retuning: add each mode's extrapolated unconverged shift (see Convergence)
# to its HFSS frequency first. The estimate is rough; off by default.
retune_with_extrapolation = False

EC_GHZ_FF = 19.37  # e^2 / 2h, in GHz * fF

## Load the runs

In [ ]:
if results_dir is None:
    found = sorted(glob.glob("hfss_results*/qubits_couplers"), key=os.path.getmtime)
    assert found, "No hfss_results*/qubits_couplers folder next to this notebook."
    results_dir = found[-1]
print(f"Results: {os.path.abspath(results_dir)}")

junctions_csv = os.path.join(results_dir, "junctions.csv")
if os.path.exists(junctions_csv):
    j = pd.read_csv(junctions_csv, index_col=0)
    design = pd.DataFrame({"target (GHz)": j["freq_ghz"], "finger (um)": j["finger_um"],
                           "C_sigma (fF)": j["csigma_ff"], "C_j (fF)": j["cj_ff"],
                           "E_J (GHz)": j["ej_ghz"]})
    print("Design values from junctions.csv")
else:
    design = pd.DataFrame(design_fallback, index=DESIGN_COLUMNS).T
    print("No junctions.csv: design values from design_fallback in Settings")


def load_run(run):
    """Mode table and cross-Kerr matrix of one run, the elements in chain order."""
    modes = pd.read_csv(os.path.join(results_dir, f"{run}_modes.csv"), index_col=0)
    chi = pd.read_csv(os.path.join(results_dir, f"{run}_chi.csv"), index_col=0)
    names = run.split("_")
    return names, modes, chi


runs, skipped = {}, {}
for path in sorted(glob.glob(os.path.join(results_dir, "*_modes.csv"))):
    run = os.path.basename(path)[: -len("_modes.csv")]
    names, modes, chi = load_run(run)
    if len(names) != 3 or set(names) != set(modes.index):
        skipped[run] = "not a three-element run with modes named after its elements"
        continue
    if chi.loc[names, names].isna().any().any():
        skipped[run] = "cross-Kerr matrix has blanks"
        continue
    low = modes.index[modes["max junction participation"] < min_participation]
    if len(low):
        skipped[run] = (f"mixed modes ({', '.join(low)}: largest junction participation "
                        f"{', '.join(f'{p:.2f}' for p in modes.loc[low, 'max junction participation'])})")
        continue
    runs[run] = dict(names=names, modes=modes, chi=chi)

# Runs saved as the mirror image of another (hfss_use_mirror_symmetry) hold
# the same numbers relabelled, in reversed chain order; fit each set once.
def fingerprint(r):
    m, c, n = r["modes"], r["chi"], r["names"]
    return tuple(np.round(np.r_[np.sort(m.loc[n, "f (GHz)"].values),
                                np.sort(c.loc[n, n].values.ravel())], 9))

seen = {}
for run, r in runs.items():
    key = fingerprint(r)
    r["mirror_of"] = seen.get(key)
    seen.setdefault(key, run)

print(f"\n{len(runs)} runs:")
for run, r in runs.items():
    m = r["modes"]
    detuned = [n for n in r["names"]
               if abs(m.loc[n, "target (GHz)"] - design.loc[n, "target (GHz)"]) > 1e-4]
    r["detuned"] = detuned
    note = f"   (mirror image of {r['mirror_of']})" if r["mirror_of"] else ""
    note += f"   (L_J detuned in this run: {', '.join(detuned)})" if detuned else ""
    print(f"  {run}{note}")
for run, why in skipped.items():
    print(f"  skipped {run}: {why}")

## The model

Each element is a tunable transmon (symmetric SQUID) at its fitted E_J and E_C, coupled to the others through their charges:

H = sum_i [4 E_C,i (n_i - n_g)^2 - E_J,i |cos(pi Phi_i / Phi_0)| cos(phi_i)] + sum_(i<j) G_ij n_i n_j

scqubits diagonalises each transmon; the coupled Hamiltonian is built in the product of their lowest levels and diagonalised here. A capacitive network gives every G_ij >= 0, which the fit enforces. The coupling quoted as **g** is the usual exchange coupling, G_ij <0|n_i|1> <0|n_j|1>.

Measured quantities, as pyEPR defines them (chi in its `_chi.csv`, MHz): anharmonicity alpha = -chi_ii, static ZZ between two modes = E_11 - E_10 - E_01 + E_00 = -chi_ij.

In [ ]:
PAIRS = [(0, 1), (1, 2), (0, 2)]
LABELS = list(itertools.product(range(levels_kept), repeat=3))


def bare(ej, ec, flux=0.0):
    """Levels (GHz, from the ground state) and charge matrix of one transmon,
    in its lowest levels_kept states."""
    t = scq.TunableTransmon(EJmax=ej, EC=ec, d=0.0, flux=flux, ng=0.0, ncut=ncut,
                            truncated_dim=levels_kept)
    evals, evecs = t.eigensys(evals_count=levels_kept)
    n = t.matrixelement_table("n_operator", evecs=evecs, evals_count=levels_kept)
    return evals - evals[0], n


def f01_alpha(ej, ec, flux=0.0):
    e, _ = bare(ej, ec, flux)
    return e[1], e[2] - 2 * e[1]


def hamiltonian(parts, G):
    """Coupled Hamiltonian in the product of the three transmons' kept levels;
    parts = [bare(...)] of each, G = couplings of PAIRS (GHz)."""
    eye = np.eye(levels_kept)

    def wrap(i, op):
        ops = [eye, eye, eye]
        ops[i] = op
        return np.kron(np.kron(ops[0], ops[1]), ops[2])

    H = sum(wrap(i, np.diag(parts[i][0])) for i in range(3)).astype(complex)
    for (i, j), g in zip(PAIRS, G):
        H += g * wrap(i, parts[i][1]) @ wrap(j, parts[j][1])
    return H


def coupled(ej, ec, G, flux=(0.0, 0.0, 0.0)):
    """Dressed energies of the three-element model, by bare label.

    Returns pick(label) -> (dressed energy, how much of the bare state that
    dressed state holds), and each transmon's bare levels and charge matrix."""
    parts = [bare(ej[i], ec[i], flux[i]) for i in range(3)]
    E, V = np.linalg.eigh(hamiltonian(parts, G))
    weight = np.abs(V) ** 2  # [bare index, dressed index]
    index = {lab: k for k, lab in enumerate(LABELS)}

    def pick(lab):
        row = weight[index[lab]]
        k = int(np.argmax(row))
        return E[k], row[k]

    return pick, parts


def observables(ej, ec, G, flux=(0.0, 0.0, 0.0)):
    """Dressed f01 and anharmonicity of each element and the ZZ of each pair
    (GHz), and the smallest overlap among the states used."""
    pick, parts = coupled(ej, ec, G, flux)
    used = {}

    def E(*lab):
        if lab not in used:
            used[lab] = pick(lab)
        return used[lab][0]

    unit = lambda i, k=1: tuple(k if m == i else 0 for m in range(3))
    e0 = E(0, 0, 0)
    f = np.array([E(*unit(i)) - e0 for i in range(3)])
    alpha = np.array([E(*unit(i, 2)) - 2 * E(*unit(i)) + e0 for i in range(3)])
    zz = np.array([E(*tuple(1 if m in p else 0 for m in range(3))) - E(*unit(p[0]))
                   - E(*unit(p[1])) + e0 for p in PAIRS])
    overlap = min(o for _, o in used.values())
    return f, alpha, zz, overlap, parts


def exchange_g(ej, ec, G, flux=(0.0, 0.0, 0.0)):
    """Exchange couplings g (GHz) of the three pairs: G_ij <0|n_i|1> <0|n_j|1>."""
    n01 = [abs(bare(ej[i], ec[i], flux[i])[1][0, 1]) for i in range(3)]
    return np.array([G[k] * n01[i] * n01[j] for k, (i, j) in enumerate(PAIRS)])

## Fit each run

Start: each element alone fitted to its own frequency and anharmonicity, and each coupling from the cross-Kerr inversion that `qubits_couplers.ipynb` uses. Then all nine parameters together, to all nine numbers.

In [ ]:
def run_data(r):
    names, modes, chi = r["names"], r["modes"], r["chi"]
    f = np.array([modes.loc[n, "f (GHz)"] for n in names])
    alpha = np.array([-chi.loc[n, n] / 1e3 for n in names])
    zz = np.array([-chi.loc[names[i], names[j]] / 1e3 for i, j in PAIRS])
    return f, alpha, zz


def fit_single(f, alpha):
    def res(x):
        f1, a1 = f01_alpha(*x)
        return [(f1 - f) * 1e3, (a1 - alpha) * 1e3]
    ec0 = -alpha
    return least_squares(res, [(f + ec0) ** 2 / (8 * ec0), ec0]).x


def fit_run(r):
    f_d, a_d, zz_d = run_data(r)
    single = np.array([fit_single(f, a) for f, a in zip(f_d, a_d)])
    n01 = [abs(bare(*single[i])[1][0, 1]) for i in range(3)]
    G0 = []
    for k, (i, j) in enumerate(PAIRS):
        d, ai, aj = f_d[i] - f_d[j], a_d[i], a_d[j]
        g = np.sqrt(abs(zz_d[k] * (d + ai) * (d - aj) / (2 * (ai + aj))))
        G0.append(max(g if (i, j) != (0, 2) else 0.2 * g, 1e-4) / (n01[i] * n01[j]))
    x0 = np.r_[single[:, 0], single[:, 1], G0]
    zz_scale = np.maximum(np.abs(zz_d), 1e-6)  # relative, floor 1 kHz

    def res(x):
        f, a, zz, _, _ = observables(x[0:3], x[3:6], x[6:9])
        return np.r_[(f - f_d) * 1e3, (a - a_d) * 1e3, (zz - zz_d) / zz_scale]

    sol = least_squares(res, x0, bounds=([1] * 3 + [0.01] * 3 + [0] * 3,
                                         [500] * 3 + [2] * 3 + [1] * 3),
                        x_scale=np.abs(x0) + 1e-4)
    ej, ec, G = sol.x[0:3], sol.x[3:6], sol.x[6:9]
    worst = np.max(np.abs(sol.fun))
    if worst > 1e-3:
        print(f"  WARNING: {r['names']}: fit misses by {worst:.3g} "
              "(MHz for f and alpha, relative for ZZ)")
    return dict(ej=ej, ec=ec, G=G, g=exchange_g(ej, ec, G), residual=worst)


fits = {}
for run, r in runs.items():
    if r["mirror_of"]:
        continue
    fits[run] = fit_run(r)
for run, r in runs.items():  # mirror images: the chain order is reversed
    if r["mirror_of"]:
        twin = fits[r["mirror_of"]]
        fits[run] = dict(ej=twin["ej"][::-1], ec=twin["ec"][::-1], G=twin["G"][[1, 0, 2]],
                         g=twin["g"][[1, 0, 2]], residual=twin["residual"])

rows = []
for run, fit in fits.items():
    names = runs[run]["names"]
    for i, n in enumerate(names):
        f_b, a_b = f01_alpha(fit["ej"][i], fit["ec"][i])
        rows.append({"run": run, "element": n, "E_J (GHz)": fit["ej"][i],
                     "E_J / design": fit["ej"][i] / design.loc[n, "E_J (GHz)"],
                     "E_C (MHz)": fit["ec"][i] * 1e3, "E_J/E_C": fit["ej"][i] / fit["ec"][i],
                     "C_sigma (fF)": EC_GHZ_FF / fit["ec"][i],
                     "bare f01 (GHz)": f_b, "bare alpha (MHz)": a_b * 1e3})
element_fits = pd.DataFrame(rows)

rows = []
for run, fit in fits.items():
    names = runs[run]["names"]
    f_d, a_d, zz_d = run_data(runs[run])
    for k, (i, j) in enumerate(PAIRS):
        d, ai, aj = (f_d[i] - f_d[j]) * 1e3, a_d[i] * 1e3, a_d[j] * 1e3
        g_chi = np.sqrt(abs(zz_d[k] * 1e3 * (d + ai) * (d - aj) / (2 * (ai + aj))))
        rows.append({"run": run, "pair": f"{names[i]}-{names[j]}",
                     "kind": "direct" if (i, j) == (0, 2) else "neighbours",
                     "g (MHz)": fit["g"][k] * 1e3,
                     "g from chi inversion (MHz)": g_chi if (i, j) != (0, 2) else np.nan,
                     "ZZ in HFSS (kHz)": zz_d[k] * 1e6})
couplings = pd.DataFrame(rows)

print("Fitted elements (model E_J, E_C; C_sigma = e^2 / 2h E_C). E_J / design: against the")
print("junction cell's E_J, how far the effective model sits from the junction itself.")
print(element_fits.round(3).to_string(index=False))
print("\nCouplings. 'direct': the two outer elements of the run, not through the middle one.")
print("'g from chi inversion': the lowest-order formula qubits_couplers.ipynb uses, for comparison.")
print(couplings.round(3).to_string(index=False))

## Coupler flux sweeps (Q-TC-Q runs)

The coupler's E_J goes as |cos(pi Phi / Phi_0)|; the qubits stay at zero flux. For each point: the dressed frequencies, the static ZZ between the qubits, and the effective qubit-qubit coupling g_eff two ways:

* **formula** (signed): g_eff = g_ab + (g_a g_b / 2) sum_(q = a, b) [1/(f_q - f_c) - 1/(f_q + f_c)], from the bare model frequencies (Yan et al. 2018, with the direct coupling g_ab added).
* **numerical** (magnitude): half the smallest splitting between the two qubit states with both qubits brought to their mean frequency, the coupler held. This is the exchange rate a gate between the two qubits would see, with no approximation in the couplings.

Points where the coupler crosses a qubit (labels unreliable) are blank.

In [ ]:
def geff_formula(ej, ec, g, flux_c):
    fa = f01_alpha(ej[0], ec[0])[0]
    fc = f01_alpha(ej[1], ec[1], flux_c)[0]
    fb = f01_alpha(ej[2], ec[2])[0]
    ga, gb, gab = g
    return gab + ga * gb / 2 * sum(1 / (fq - fc) - 1 / (fq + fc) for fq in (fa, fb))


def geff_numeric(ej, ec, G, flux_c):
    """Half the minimum splitting of the qubit-a / qubit-b single excitations,
    with qubit a retuned (E_J only) to the qubits' mean bare frequency and
    qubit b swept through it."""
    a, b = LABELS.index((1, 0, 0)), LABELS.index((0, 0, 1))
    f_mid = (f01_alpha(ej[0], ec[0])[0] + f01_alpha(ej[2], ec[2])[0]) / 2
    ej_a = brentq(lambda x: f01_alpha(x, ec[0])[0] - f_mid, 0.3 * ej[0], 3 * ej[0])
    fixed = [bare(ej_a, ec[0]), bare(ej[1], ec[1], flux_c)]

    def splitting(ej_b):
        E, V = np.linalg.eigh(hamiltonian(fixed + [bare(ej_b, ec[2])], G))
        w = np.abs(V[a]) ** 2 + np.abs(V[b]) ** 2
        k1, k2 = np.argsort(w)[-2:]
        return abs(E[k1] - E[k2])

    ej_res = brentq(lambda x: f01_alpha(x, ec[2])[0] - f_mid, 0.3 * ej[2], 3 * ej[2])
    best = minimize_scalar(splitting, bounds=(0.97 * ej_res, 1.03 * ej_res), method="bounded",
                           options=dict(xatol=1e-5))
    return best.fun / 2


def crossings(x, y):
    """x where y changes sign (linear interpolation)."""
    y = np.asarray(y, dtype=float)
    out = []
    for k in range(len(y) - 1):
        if np.isfinite(y[k]) and np.isfinite(y[k + 1]) and y[k] * y[k + 1] < 0:
            out.append(x[k] - y[k] * (x[k + 1] - x[k]) / (y[k + 1] - y[k]))
    return out


sweeps, idle_rows = {}, []
for run, fit in fits.items():
    names = runs[run]["names"]
    if not names[1].startswith("TC") or runs[run]["mirror_of"]:
        continue
    ej, ec, G, g = fit["ej"], fit["ec"], fit["G"], fit["g"]
    rows = []
    for phi in flux_points:
        f, alpha, zz, overlap, _ = observables(ej, ec, G, (0.0, phi, 0.0))
        ok = overlap >= min_label_overlap
        rows.append({"flux": phi, f"f {names[0]}": f[0], f"f {names[1]}": f[1],
                     f"f {names[2]}": f[2],
                     "ZZ (kHz)": zz[2] * 1e6 if ok else np.nan,
                     "g_eff formula (MHz)": geff_formula(ej, ec, g, phi) * 1e3 if ok else np.nan})
    sweep = pd.DataFrame(rows).set_index("flux")
    num = {}
    for phi in geff_flux_points:
        _, _, _, overlap, _ = observables(ej, ec, G, (0.0, phi, 0.0))
        num[phi] = geff_numeric(ej, ec, G, phi) * 1e3 if overlap >= min_label_overlap else np.nan
    sweeps[run] = (sweep, pd.Series(num, name="|g_eff| numerical (MHz)"))

    idle = sweep.iloc[0]
    zz_zero = crossings(sweep.index.values, sweep["ZZ (kHz)"].values)
    g_zero = crossings(sweep.index.values, sweep["g_eff formula (MHz)"].values)
    idle_rows.append({
        "run": run, "coupler": names[1],
        "f coupler idle (GHz)": idle[f"f {names[1]}"],
        "ZZ idle (kHz)": idle["ZZ (kHz)"],
        "g_eff idle, formula (MHz)": idle["g_eff formula (MHz)"],
        "|g_eff| idle, numerical (MHz)": num[geff_flux_points[0]],
        "g direct (MHz)": g[2] * 1e3,
        "ZZ = 0 at flux": ", ".join(f"{x:.3f}" for x in zz_zero) or "none in sweep",
        "g_eff = 0 at flux": ", ".join(f"{x:.3f}" for x in g_zero) or "none in sweep",
    })

idle_summary = pd.DataFrame(idle_rows).set_index("run")
print("At the idle point (coupler at zero flux) and the zero crossings in the sweep:")
print(idle_summary.round(3).to_string())

for run, (sweep, num) in sweeps.items():
    names = runs[run]["names"]
    fig, ax = plt.subplots(1, 3, figsize=(14, 3.6))
    for n in names:
        ax[0].plot(sweep.index, sweep[f"f {n}"], label=n)
    ax[0].set(xlabel="coupler flux (Phi_0)", ylabel="dressed f01 (GHz)", title=f"{run}: frequencies")
    ax[0].legend(fontsize=8)
    ax[1].plot(sweep.index, sweep["ZZ (kHz)"])
    ax[1].axhline(0, color="0.6", lw=0.8)
    ax[1].set(xlabel="coupler flux (Phi_0)", ylabel="ZZ (kHz)", title="static ZZ",
              yscale="symlog")
    ax[2].plot(sweep.index, sweep["g_eff formula (MHz)"], label="formula (signed)")
    ax[2].plot(num.index, num.values, "o", label="numerical |g_eff|")
    ax[2].axhline(0, color="0.6", lw=0.8)
    ax[2].set(xlabel="coupler flux (Phi_0)", ylabel="g_eff (MHz)", title="qubit-qubit coupling")
    ax[2].legend(fontsize=8)
    fig.tight_layout()
    plt.show()

## Coupler-qubit-coupler runs

In [ ]:
rows = []
for run, fit in fits.items():
    names = runs[run]["names"]
    if not names[1].startswith("Q"):
        continue
    f_d, a_d, zz_d = run_data(runs[run])
    rows.append({"run": run, "qubit": names[1], "TCa": names[0], "TCb": names[2],
                 "g TCa-Q (MHz)": fit["g"][0] * 1e3,
                 "g Q-TCb (MHz)": fit["g"][1] * 1e3,
                 "g coupler-coupler direct (MHz)": fit["g"][2] * 1e3,
                 "ZZ coupler-coupler (kHz)": zz_d[2] * 1e6,
                 "detuned in run": ", ".join(runs[run]["detuned"]) or "-"})
if rows:
    print("The two couplers either side of a qubit (TCa: the lower-numbered one):")
    print(pd.DataFrame(rows).set_index("run").round(3).to_string())

## Convergence of each run

From `<run>_convergence.csv`: the frequency of each mode (the linear HFSS frequency, so above the f01 in the tables) over the last adaptive passes. Where the last steps all go the same way and shrink, the remaining shift is estimated from their ratio r as d_last r / (1 - r). A rough estimate, but it shows which runs stopped early.

In [ ]:
conv_rows = []
for run in runs:
    path = os.path.join(results_dir, f"{run}_convergence.csv")
    if not os.path.exists(path):
        continue
    conv = pd.read_csv(path, header=[0, 1], index_col=0).dropna(how="all")
    for name in runs[run]["names"]:
        if (name, "f (GHz)") not in conv:
            continue
        f = conv[(name, "f (GHz)")].dropna().values * 1e3
        d = np.diff(f)
        row = {"run": run, "element": name, "passes": len(f),
               "last step (MHz)": d[-1] if len(d) else np.nan}
        if len(d) >= 5 and (np.all(d[-4:] > 0) or np.all(d[-4:] < 0)):
            r_ = float(np.clip(np.median(d[-4:] / d[-5:-1]), 0, 0.95))
            row["est. remaining shift (MHz)"] = d[-1] * r_ / (1 - r_)
        else:
            row["est. remaining shift (MHz)"] = 0.0
        if (name, "Q") in conv and conv[(name, "Q")].notna().any():
            q = conv[(name, "Q")].dropna().values
            row["last Q change (%)"] = abs(q[-1] / q[-2] - 1) * 100 if len(q) > 1 else np.nan
        conv_rows.append(row)
convergence = pd.DataFrame(conv_rows)
if len(convergence):
    print(convergence.round(2).to_string(index=False))
    bad = convergence[convergence["est. remaining shift (MHz)"].abs() > 2]
    if len(bad):
        print(f"\n{len(bad)} modes may still move by more than 2 MHz: "
              + ", ".join(f"{e} in {r}" for r, e in zip(bad["run"], bad["element"])))

t1_rows = [{"run": run, "element": n, "Q": r["modes"].loc[n, "Q"],
            "T1 limit (us)": r["modes"].loc[n, "T1 limit (us)"]}
           for run, r in runs.items() if "T1 limit (us)" in r["modes"] for n in r["names"]]
if t1_rows:
    print("\nControl-line T1 limits (from each mode's Q):")
    print(pd.DataFrame(t1_rows).round(1).to_string(index=False))
else:
    print("\nNo Q in these runs (no lossy ports): no T1 limits.")

## Retuning

For each element: its HFSS frequency (averaged over the runs it was in, leaving out runs where its L_J was detuned), the target, and two ways to move it there with the fitted model. The shift each element gets from its neighbours (dressed minus bare f01, from the fits) is held fixed.

* **C_sigma**: same junction, E_C changed. The only option for the qubits, whose finger is at the DFM width. Also given: the HFSS calibration factor that would make `qubits_couplers.ipynb` size the pad for that C_sigma (one factor per element type, so the per-element factors are averaged; they should agree to a percent or so).
* **Junction**: same pad, E_J changed, as a finger width on the 5 nm grid (Jc is fixed, so E_J is proportional to the finger width).

Changing C_sigma by a few percent changes g by about half as much; re-simulate once after retuning to confirm.

In [ ]:
rows = []
for name in design.index:
    f_runs, shifts, alpha_shifts, ejs, ecs = [], [], [], [], []
    for run, fit in fits.items():
        names = runs[run]["names"]
        if name not in names or name in runs[run]["detuned"]:
            continue
        i = names.index(name)
        f_d, a_d, _ = run_data(runs[run])
        f_b, a_b = f01_alpha(fit["ej"][i], fit["ec"][i])
        extra = 0.0
        if retune_with_extrapolation and len(convergence):
            m = convergence[(convergence.run == run) & (convergence.element == name)]
            extra = float(m["est. remaining shift (MHz)"].iloc[0]) / 1e3 if len(m) else 0.0
        f_runs.append(f_d[i] + extra)
        shifts.append(f_d[i] - f_b)
        alpha_shifts.append(a_d[i] - a_b)
        ejs.append(fit["ej"][i])
        ecs.append(fit["ec"][i])
    if not f_runs:
        continue
    ej, ec = np.mean(ejs), np.mean(ecs)
    shift, a_shift = np.mean(shifts), np.mean(alpha_shifts)
    target = design.loc[name, "target (GHz)"]
    f_bare_needed = target - shift
    # C_sigma route
    ec_new = brentq(lambda x: f01_alpha(ej, x)[0] - f_bare_needed, 0.3 * ec, 3 * ec)
    c_now, c_new = EC_GHZ_FF / ec, EC_GHZ_FF / ec_new
    # junction route
    ej_new = brentq(lambda x: f01_alpha(x, ec)[0] - f_bare_needed, 0.3 * ej, 3 * ej)
    finger = design.loc[name, "finger (um)"]
    finger_new = round(round(finger * ej_new / ej / jj_grid_um) * jj_grid_um, 6)
    # Calibration factor: qubits_couplers.ipynb sizes each pad so that
    # C_table * factor + C_j = C_design. In HFSS (as the model sees it) that pad
    # gives c_now - C_j; to give c_new - C_j instead, the factor shrinks by
    # (c_now - C_j) / (c_new - C_j).
    cj = design.loc[name, "C_j (fF)"]
    factor_old = qubit_csigma_hfss_factor if name.startswith("Q") else coupler_csigma_hfss_factor
    rows.append({
        "element": name, "runs": len(f_runs),
        "f HFSS (GHz)": np.mean(f_runs), "spread (MHz)": (max(f_runs) - min(f_runs)) * 1e3,
        "target (GHz)": target, "off by (MHz)": (np.mean(f_runs) - target) * 1e3,
        "alpha now (MHz)": (f01_alpha(ej, ec)[1] + a_shift) * 1e3,
        "C_sigma change (fF)": c_new - c_now,
        "C_sigma change (%)": (c_new / c_now - 1) * 100,
        "alpha with C change (MHz)": (f01_alpha(ej, ec_new)[1] + a_shift) * 1e3,
        "calibration factor for it": factor_old * (c_now - cj) / (c_new - cj),
        "finger now (um)": finger, "finger for target (um)": finger_new,
        "alpha with finger change (MHz)": (f01_alpha(finger_new / finger * ej, ec)[1] + a_shift) * 1e3,
    })
retune = pd.DataFrame(rows).set_index("element")
print(retune.round(3).to_string())
for kind, old in (("Q", qubit_csigma_hfss_factor), ("TC", coupler_csigma_hfss_factor)):
    sel = retune[retune.index.str.startswith(kind)]
    if len(sel):
        fac = sel["calibration factor for it"]
        label = "qubit" if kind == "Q" else "coupler"
        print(f"\n{label}_csigma_hfss_factor: {old} -> {fac.mean():.4f} "
              f"(per element {fac.min():.4f}-{fac.max():.4f})")
low = retune.index[retune["finger for target (um)"] < dfm_finger_um]
if len(low):
    print(f"Finger route under the {dfm_finger_um} um DFM width for: {', '.join(low)}; use C_sigma.")

## Save

In [ ]:
out = os.path.join(results_dir, "scqubits")
os.makedirs(out, exist_ok=True)
element_fits.to_csv(os.path.join(out, "element_fits.csv"), index=False)
couplings.to_csv(os.path.join(out, "couplings.csv"), index=False)
if len(idle_rows):
    idle_summary.to_csv(os.path.join(out, "coupler_idle_summary.csv"))
for run, (sweep, num) in sweeps.items():
    sweep.join(num, how="outer").to_csv(os.path.join(out, f"{run}_flux_sweep.csv"))
if len(convergence):
    convergence.to_csv(os.path.join(out, "convergence_summary.csv"), index=False)
retune.to_csv(os.path.join(out, "retune.csv"))
print(f"Saved to {os.path.abspath(out)}")